In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow import keras
from keras.optimizers import AdamW
from keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns


## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Load paths from .env file
dataset_dir = os.getenv("DATASET_DIR")
datasplits_dir = os.getenv("DATASPLITS_DIR")

# Patgs to the data splits files
train_file = os.path.join(datasplits_dir, "img_train.tsv")
val_file = os.path.join(datasplits_dir, "img_val.tsv")
test_file = os.path.join(datasplits_dir, "img_test.tsv")

# Load the data splits into pandas DataFrames
train_df = pd.read_csv(train_file, sep="\t")
val_df = pd.read_csv(val_file, sep="\t")
test_df = pd.read_csv(test_file, sep="\t")

In [ ]:
train_df.shape

In [ ]:
train_df.head(1)

In [ ]:
val_df.shape

In [ ]:
val_df.head(1)

In [ ]:
test_df.shape

In [ ]:
test_df.head(1)

## Label Encoding

In [ ]:
label_encoders = {}
for col in ["image_info", "image_human"]:
	le = LabelEncoder()
	all_values = pd.concat([train_df[col], val_df[col], test_df[col]])
	le.fit(all_values)
	train_df[col] = le.transform(train_df[col])
	val_df[col] = le.transform(val_df[col])
	test_df[col] = le.transform(test_df[col])
	label_encoders[col] = le


## Helper Functions

In [ ]:
def get_backbone(name, input_shape):
	if name == "ResNet50":
		return keras.applications.ResNet50(input_shape=input_shape, include_top=False, weights="imagenet"), keras.applications.resnet50.preprocess_input
	elif name == "VGG16":
		return keras.applications.VGG16(input_shape=input_shape, include_top=False, weights="imagenet"), keras.applications.vgg16.preprocess_input
	elif name == "EfficientNetB0":
		return keras.applications.EfficientNetB0(input_shape=input_shape, include_top=False, weights="imagenet"), keras.applications.efficientnet.preprocess_input
	else:
		raise ValueError(f"Unknown model name: {name}")

In [ ]:
def load_image_factory(preprocess_fn, image_size):
	def load_image(path, label1, label2):
		img = tf.io.read_file(path)
		img = tf.image.decode_jpeg(img, channels=3)
		img = tf.image.resize(img, image_size)
		img = preprocess_fn(img)
		return img, {"info": label1, "human": label2}
	return load_image

In [ ]:
def df_to_dataset(df, load_image_fn, batch_size=32, shuffle=True):
	full_paths = df["image_path"].apply(lambda p: os.path.normpath(os.path.join(dataset_dir, p))).values
	labels1 = df["image_info"].values.astype("float32")  # for sigmoid binary output
	labels2 = df["image_human"].values
	image_ds = tf.data.Dataset.from_tensor_slices((full_paths, labels1, labels2))
	image_ds = image_ds.map(load_image_fn, num_parallel_calls=tf.data.AUTOTUNE)
	if shuffle:
		image_ds = image_ds.shuffle(700)
	return image_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
def build_full_model(base_model, image_size, num_human_classes, dropout_rate=0.6, classifier_units=128):
	inputs = keras.Input(shape=(*image_size, 3))

	data_augmentation = keras.Sequential([
		keras.layers.RandomFlip("horizontal"),
		keras.layers.RandomRotation(0.1),
		keras.layers.RandomZoom(0.2),
	])

	x = data_augmentation(inputs)
	
	x = base_model(x)
	
	x = keras.layers.GlobalAveragePooling2D()(x)
	x = keras.layers.BatchNormalization()(x)
	x = keras.layers.Dense(classifier_units)(x)
	x = keras.layers.Activation("relu")(x)
	x = keras.layers.Dropout(dropout_rate)(x)

	info_out = keras.layers.Dense(1, activation="sigmoid", name="info")(x)
	human_out = keras.layers.Dense(num_human_classes, activation="softmax", name="human")(x)

	return keras.Model(inputs=inputs, outputs=[info_out, human_out])


In [ ]:
def plot_training_history(history):
	for key in history.history:
		if not key.startswith("val_"):
			plt.figure()
			plt.plot(history.history[key], label="train")
			plt.plot(history.history[f"val_{key}"], label="val")
			plt.title(key)
			plt.xlabel("Epoch")
			plt.ylabel("Value")
			plt.legend()
			plt.grid(True)
			plt.show()

In [ ]:
def partial_unfreeze(base_model, model, model_name):
    base_model.trainable = True  # Start by enabling training globally

    for layer in base_model.layers:
        # Handle ResNet50
        if model_name == "ResNet50":
            if layer.name.startswith(("conv5_block2", "conv5_block3")):
                layer.trainable = not isinstance(layer, keras.layers.BatchNormalization)
            else:
                layer.trainable = False

        # Handle VGG16
        elif model_name == "VGG16":
            if layer.name.startswith("block5"):
                layer.trainable = True
            else:
                layer.trainable = False

        # Handle EfficientNetB0
        elif model_name == "EfficientNetB0":
            # All layers are fine to unfreeze except BatchNorm
            layer.trainable = not isinstance(layer, keras.layers.BatchNormalization)

        else:
            raise ValueError(f"Unknown model name: {model_name}")

    # Summary log
    print("Unfrozen layers:")
    for layer in base_model.layers:
        if layer.trainable:
            print(f"  {layer.name} ({layer.__class__.__name__})")

    print(f"\nTrainable {model_name.upper()} layers after partial unfreezing: {sum([layer.trainable for layer in base_model.layers])}")
    print(f"Trainable {model_name.upper()} weights after partial unfreezing: {len(base_model.trainable_weights)}")
    print(f"Overall trainable weights: {len(model.trainable_weights)}\n")


In [ ]:
def evaluate_task(y_true, y_pred, task_name, class_names=None):
	print(f"\n{task_name.upper()} classification report:")

	print(classification_report(y_true, y_pred, target_names=class_names))

	acc = accuracy_score(y_true, y_pred)
	prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
	rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
	f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

	print(f"> Accuracy:  {acc:.4f}")
	print(f"> Precision: {prec:.4f}")
	print(f"> Recall:    {rec:.4f}")
	print(f"> F1-score:  {f1:.4f}")

	# Shorten class names only for "image_human" task
	if task_name == "image_human":
		display_names = [c[0].upper() for c in class_names]
	else:
		display_names = class_names

	cm = confusion_matrix(y_true, y_pred)
	plt.figure(figsize=(6, 5))
	sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
				xticklabels=display_names, yticklabels=display_names)
	plt.title(f"{task_name.upper()} - Confusion Matrix")
	plt.xlabel("Predicted")
	plt.ylabel("True")
	plt.tight_layout()
	plt.show()


In [ ]:
def evaluate_model(model, test_ds, model_name):
    y_true_info, y_pred_info = [], []
    y_true_human, y_pred_human = [], []

    for img_batch, label_batch in tqdm(test_ds, desc="Running inference on test set"):
        # Predict outputs for each task
        preds = model.predict(img_batch, verbose=0)
        # Task 1: informativeness (binary - sigmoid)
        y_true_info.extend(label_batch["info"].numpy())
        y_pred_info.extend((preds[0] > 0.5).astype("int32").flatten())
        # Task 2: humanitarian category (multiclass - softmax)
        y_true_human.extend(label_batch["human"].numpy())
        y_pred_human.extend(np.argmax(preds[1], axis=1))

    evaluate_task(
	    y_true_info,
	    y_pred_info,
	    "image_info",
	    class_names=["not informative", "informative"]
    )

    evaluate_task(
	    y_true_human,
	    y_pred_human,
	    "image_human",
	    class_names=label_encoders["image_human"].classes_
    )
    


## Hyperparameters

In [ ]:
IMAGE_SIZE = (224, 224)

BATCH_SIZE = 32

LR_TRAIN = 1e-4
WD_TRAIN = 1e-5
EPOCHS_TRAIN = 100
PATIENCE_TRAIN = 5

LR_FT = 1e-6
WD_FT = 1e-5
EPOCHS_FT = 20
PATIENCE_FT = 3

## Mutlitmodel evaluation pipeline

In [ ]:
backbones = ["ResNet50", "VGG16", "EfficientNetB0"]

for model_name in backbones:
    print(f"\nStarting pipeline for: {model_name.upper()}\n\n")

    # 1. Load base model + preprocessing
    base_model, preprocess_fn = get_backbone(model_name, input_shape=(*IMAGE_SIZE, 3))

    # 2. Prepare data loaders with preprocess_fn
    load_image = load_image_factory(preprocess_fn, IMAGE_SIZE)
    train_ds = df_to_dataset(train_df, load_image, BATCH_SIZE, shuffle=True)
    val_ds = df_to_dataset(val_df, load_image, BATCH_SIZE)
    test_ds = df_to_dataset(test_df, load_image, BATCH_SIZE, shuffle=False)

    # 3. Build model
    base_model.trainable = False
    model = build_full_model(base_model, IMAGE_SIZE, num_human_classes=train_df["image_human"].nunique())

    # 4. Compile
    model.compile(
        optimizer=AdamW(LR_TRAIN, weight_decay=WD_TRAIN),
        loss={"info": "binary_crossentropy", "human": "sparse_categorical_crossentropy"},
        metrics={"info": "accuracy", "human": "accuracy"}
    )

    # 5. Train (frozen)
    print(f"\nTraining {model_name.upper()} model...")
    callbacks = [
        ModelCheckpoint(f"./models/{model_name}_trained.keras", save_best_only=True, monitor="val_loss", verbose=0),
        EarlyStopping(patience=PATIENCE_TRAIN, restore_best_weights=True)
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TRAIN, callbacks=callbacks)
    plot_training_history(history)

    # 6. Fine-tune
    print(f"\nFine-tuning {model_name.upper()} model...")
    partial_unfreeze(base_model, model, model_name)
    model.compile(
        optimizer=AdamW(LR_FT, weight_decay=WD_FT),
        loss={"info": "binary_crossentropy", "human": "sparse_categorical_crossentropy"},
        metrics={"info": "accuracy", "human": "accuracy"}
    )
    callbacks = [
        ModelCheckpoint(f"./models/{model_name}_fine_tuned.keras", save_best_only=True, monitor="val_loss", verbose=0),
        EarlyStopping(patience=PATIENCE_FT, restore_best_weights=True)
    ]
    ft_history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, callbacks=callbacks)
    plot_training_history(ft_history)

    # 7. Evaluate
    print(f"\nEvaluating {model_name.upper()} model...")
    evaluate_model(model, test_ds, model_name)
